# ARCH 6133 Places / Platforms  
## Points of Interest Data: Platform Definitions of Place

> **Open in Colab badge**  
> Once this notebook is hosted on GitHub, replace the placeholders below with your repository path.  
>
> `[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPO/blob/main/ARCH6133_POI_Data_Tutorial_COLAB_READY.ipynb)`
>
> Example Colab URL pattern:  
> `https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPO/blob/main/path/to/notebook.ipynb`

---


## Documentation links

Keep these open as you work:

### Google Places API
- Overview: https://developers.google.com/maps/documentation/places/web-service/overview
- Nearby Search: https://developers.google.com/maps/documentation/places/web-service/nearby-search
- Text Search: https://developers.google.com/maps/documentation/places/web-service/text-search
- Place Details: https://developers.google.com/maps/documentation/places/web-service/place-details
- Place Data Fields: https://developers.google.com/maps/documentation/places/web-service/data-fields
- Place Types: https://developers.google.com/maps/documentation/places/web-service/place-types
- Popular times support note: https://support.google.com/business/answer/6263531?hl=en

### Foursquare
- Place Search: https://docs.foursquare.com/developer/reference/place-search
- Response Fields: https://docs.foursquare.com/developer/reference/response-fields
- Places Pro and Premium schema, including `hours_popular`: https://docs.foursquare.com/data-products/docs/places-pro-and-premium

### OpenStreetMap
- Map features: https://wiki.openstreetmap.org/wiki/Map_features
- `amenity=*`: https://wiki.openstreetmap.org/wiki/Key:amenity
- Overpass API by example: https://wiki.openstreetmap.org/wiki/Overpass_API/Overpass_API_by_Example
- Overpass Turbo: https://overpass-turbo.eu/
- OSMnx features module: https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx-features-module

### Overture Maps
- Places Guide: https://docs.overturemaps.org/guides/places/
- Quickstart: https://docs.overturemaps.org/getting-data/
- DuckDB guide: https://docs.overturemaps.org/getting-data/duckdb/
- Place schema: https://docs.overturemaps.org/schema/reference/places/place/


## Conceptual setup: four models of place

| Source | What it thinks a place is | What it is especially useful for | What to watch critically |
|---|---|---|---|
| Google Places | A searchable and navigable destination connected to Google Maps and Search | names, addresses, hours, ratings, reviews, categories, accessibility, amenities, map links | the API exposes only a controlled subset of platform knowledge |
| Foursquare Places | A venue or POI enriched by categories, visits, check-ins, popularity, and location intelligence | popularity, popular hours, venue metadata, chains, rich categories | behavioral proxies depend on tracking infrastructures and sufficient activity |
| OpenStreetMap | A volunteered geographic feature encoded through tags | civic and infrastructural detail, open data, editability, non-commercial features | coverage and tagging are uneven and community-shaped |
| Overture Maps | An open, distributed point representation of real-world entities compiled from multiple sources | bulk data, confidence, source attribution, GeoParquet, GERS-aware IDs | source conflation and standardization require interpretation |

**Important Google caveat:** Google Maps may show popular times, live busyness, wait times, and typical visit duration for some businesses, but these are not standard public fields in the current Places API field list. Treat this as a productive gap: the API is not the platform.


## Data pipeline for today

```mermaid
flowchart LR
    A[Choose study area] --> B[Query platform sources]
    B --> C[Save raw JSON or GeoJSON]
    C --> D[Flatten source-specific fields]
    D --> E[Normalize selected fields]
    E --> F[Compare coverage and metadata]
    F --> G[Map and visualize]
    G --> H[Reflect on absences and prototype uses]
```

If your notebook environment does not render Mermaid diagrams, read the code block as a text diagram.


# 00. Setup

Run this section first. In Google Colab, uncomment the install line. If you are running locally and already have the packages installed, you can skip it.


In [ ]:
# In Google Colab, uncomment this line if needed.
# !pip -q install pandas geopandas shapely requests folium matplotlib tqdm osmnx overturemaps duckdb

from pathlib import Path
import json
import os
import time
import math
import getpass
from typing import Any, Dict, List, Optional

import pandas as pd
import requests
import folium
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

DATA_DIR = Path('poi_data')
RAW_DIR = DATA_DIR / 'raw'
OUT_DIR = DATA_DIR / 'output'
for d in [DATA_DIR, RAW_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Data folders ready:', DATA_DIR.resolve())

## API keys and access for this notebook

> **Run this section before the API tutorials.**  
> This notebook uses several different place-data sources. Some require API keys and some do not.

| Source | API key needed? | When you need it | Notes |
|---|---:|---|---|
| **Google Places API** | Yes | Google Nearby Search and Place Details cells | Requires a Google Maps Platform project, billing enabled, and the Places API enabled. Keep field masks modest to control cost. |
| **Foursquare Places API** | Yes | Foursquare Search and rich metadata cells | Some richer fields may depend on your Foursquare account, product tier, or available response fields. |
| **OpenStreetMap / Overpass** | No | OSM POI queries | Public service. Keep bounding boxes small and avoid repeated large requests. |
| **Overture Maps Places** | No | Overture CLI or DuckDB GeoParquet examples | Public open dataset. Requires internet access and package installation, but not an API key. |

### Recommended key storage

Do **not** hard-code API keys into a notebook that will be uploaded to GitHub or shared with students.

In Colab, use the **Secrets** panel in the left sidebar and add:

```text
GOOGLE_API_KEY
FOURSQUARE_API_KEY
```

Locally, you can set environment variables with the same names. As a fallback, the setup cell below will ask you to paste a key at runtime. Press Enter to skip a provider you do not plan to use.


In [ ]:
# --- API KEY SETUP ------------------------------------------------------------
# Run this once near the beginning of the notebook.
#
# This cell defines all external API keys used later in the tutorial.
# Keys are read in this order:
#   1. Environment variables, such as GOOGLE_API_KEY
#   2. Google Colab Secrets, if running in Colab
#   3. Manual prompt using getpass, so the key is not displayed in output
#
# Sources that do NOT need keys:
#   - OpenStreetMap / Overpass
#   - Overture Maps public GeoParquet data

import os
import getpass

# Optional: read from Google Colab Secrets, if available.
try:
    from google.colab import userdata  # type: ignore
except Exception:
    userdata = None


def read_secret(name: str, aliases=None):
    """
    Read a secret from environment variables or Colab Secrets.

    Parameters
    ----------
    name : str
        Primary secret name.
    aliases : list[str] | None
        Alternative names to check.

    Returns
    -------
    str | None
        The secret value, or None if not found.
    """
    candidates = [name] + (aliases or [])

    for key_name in candidates:
        value = os.environ.get(key_name)
        if value:
            return value

    if userdata is not None:
        for key_name in candidates:
            try:
                value = userdata.get(key_name)
                if value:
                    return value
            except Exception:
                pass

    return None


def prompt_for_key(label: str, existing_value=None):
    """
    Prompt for a key only if it was not already found.
    Press Enter to skip that provider.
    """
    if existing_value:
        print(f"{label}: found")
        return existing_value

    value = getpass.getpass(f"{label}: paste key or press Enter to skip: ").strip()
    if value:
        print(f"{label}: added for this runtime")
        return value

    print(f"{label}: skipped")
    return None


# Required only for the Google Places section.
GOOGLE_API_KEY = read_secret(
    "GOOGLE_API_KEY",
    aliases=["GOOGLE_PLACES_API_KEY", "GOOGLE_MAPS_API_KEY"]
)
GOOGLE_API_KEY = prompt_for_key("Google Places API key", GOOGLE_API_KEY)

# Required only for the Foursquare section.
FOURSQUARE_API_KEY = read_secret(
    "FOURSQUARE_API_KEY",
    aliases=["FSQ_API_KEY", "FOURSQUARE_PLACES_API_KEY"]
)
FOURSQUARE_API_KEY = prompt_for_key("Foursquare API key", FOURSQUARE_API_KEY)

# Convenience flags used later in the notebook.
HAS_GOOGLE_KEY = bool(GOOGLE_API_KEY)
HAS_FOURSQUARE_KEY = bool(FOURSQUARE_API_KEY)

print("\nKey status")
print("Google Places:", "ready" if HAS_GOOGLE_KEY else "not configured")
print("Foursquare:", "ready" if HAS_FOURSQUARE_KEY else "not configured")
print("OpenStreetMap / Overpass: no key required")
print("Overture Maps Places: no key required")


# 01. Define a study area

For the in-class demo, the default area is a small East Harlem / 125th Street area. You should replace this with your own study area.

Bounding box order used here:

```text
min_lon, min_lat, max_lon, max_lat
```

The center and radius are used for APIs that prefer circular searches.


In [ ]:
# Default study area: East Harlem / 125th Street area.
# Replace these with your own study area.
STUDY_AREA_NAME = 'East Harlem / 125th Street demo area'
BBOX = (-73.955, 40.795, -73.930, 40.815)  # min_lon, min_lat, max_lon, max_lat
CENTER_LAT = (BBOX[1] + BBOX[3]) / 2
CENTER_LON = (BBOX[0] + BBOX[2]) / 2
RADIUS_METERS = 900

print(STUDY_AREA_NAME)
print('BBOX:', BBOX)
print('Center:', CENTER_LAT, CENTER_LON)
print('Radius meters:', RADIUS_METERS)

m = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=15, tiles='CartoDB positron')
folium.Rectangle(bounds=[[BBOX[1], BBOX[0]], [BBOX[3], BBOX[2]]], fill=False).add_to(m)
folium.Circle(location=[CENTER_LAT, CENTER_LON], radius=RADIUS_METERS, fill=False).add_to(m)
m

## Helper functions

These functions save files, flatten nested fields, convert rows to GeoJSON, and make quick maps and charts.


In [ ]:
def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    print('Saved', path)


def load_json(path: Path) -> Any:
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def safe_get(d: Dict, keys: List[str], default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


def rows_to_geojson(rows: List[Dict], lon_col='lon', lat_col='lat') -> Dict:
    features = []
    for row in rows:
        lon, lat = row.get(lon_col), row.get(lat_col)
        if lon is None or lat is None:
            continue
        props = {k: v for k, v in row.items() if k not in [lon_col, lat_col]}
        features.append({
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [float(lon), float(lat)]},
            'properties': props
        })
    return {'type': 'FeatureCollection', 'features': features}


def save_geojson(rows: List[Dict], path: Path, lon_col='lon', lat_col='lat') -> Dict:
    gj = rows_to_geojson(rows, lon_col=lon_col, lat_col=lat_col)
    save_json(gj, path)
    return gj


def quick_map(df: pd.DataFrame, name_col='name', source_col='source', lat_col='lat', lon_col='lon'):
    m = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=15, tiles='CartoDB positron')
    folium.Rectangle(bounds=[[BBOX[1], BBOX[0]], [BBOX[3], BBOX[2]]], fill=False).add_to(m)
    for _, row in df.dropna(subset=[lat_col, lon_col]).iterrows():
        name = row.get(name_col, 'Unnamed')
        source = row.get(source_col, '')
        category = row.get('category_original', row.get('category_standardized', ''))
        popup = folium.Popup(f'<b>{name}</b><br>{source}<br>{category}', max_width=300)
        folium.CircleMarker(
            location=[row[lat_col], row[lon_col]],
            radius=4,
            popup=popup,
            fill=True
        ).add_to(m)
    return m


def plot_counts(df: pd.DataFrame, column: str, title: str, top_n=15):
    counts = df[column].fillna('missing').astype(str).value_counts().head(top_n).sort_values()
    ax = counts.plot(kind='barh', figsize=(8, max(3, 0.35 * len(counts))))
    ax.set_title(title)
    ax.set_xlabel('Count')
    ax.set_ylabel(column)
    plt.tight_layout()
    plt.show()


def first_nonempty(*vals):
    for v in vals:
        if v not in [None, '', [], {}] and not (isinstance(v, float) and math.isnan(v)):
            return v
    return None

print('Helpers ready')

# 02. Google Places API

Google Places is useful for structured place records tied to Google Maps and Search. We will use Nearby Search and optionally Place Details.

Important details:

- Nearby Search uses a POST request to `https://places.googleapis.com/v1/places:searchNearby`.
- Requests require a field mask through the `X-Goog-FieldMask` header.
- Field masks control cost, latency, and returned fields.
- Do not use `*` for class exercises unless you are intentionally testing and understand the billing implications.
- Popular times, live busyness, wait times, and typical visit duration are described in Google support documentation, but they are not standard public Places API fields in the current field list.

Before running this section, create a Google Maps Platform API key with Places API enabled.


In [ ]:
# GOOGLE_API_KEY is defined in the consolidated API key setup cell above.
GOOGLE_NEARBY_URL = 'https://places.googleapis.com/v1/places:searchNearby'
GOOGLE_DETAILS_URL = 'https://places.googleapis.com/v1/places/{place_id}'

# Keep this field mask modest for class use.
GOOGLE_NEARBY_FIELD_MASK = ','.join([
    'places.id',
    'places.displayName',
    'places.formattedAddress',
    'places.location',
    'places.primaryType',
    'places.types',
    'places.businessStatus',
    'places.rating',
    'places.userRatingCount',
    'places.priceLevel',
    'places.regularOpeningHours',
    'places.currentOpeningHours',
    'places.websiteUri',
    'places.googleMapsUri'
])

print(GOOGLE_NEARBY_FIELD_MASK)

## Google Nearby Search

The `includedTypes` list must use Google place type names. Try one or more of the following:

```text
restaurant, cafe, library, museum, school, hospital, park, supermarket, church, transit_station
```

The search below runs one request per included type, then deduplicates by Google place ID.


In [ ]:
# GOOGLE_API_KEY is defined in the consolidated API key setup cell above.
def google_nearby_search(place_type: str, max_result_count: int = 20) -> Dict:
    if not GOOGLE_API_KEY:
        raise ValueError('No Google API key provided')
    payload = {
        'includedTypes': [place_type],
        'maxResultCount': max_result_count,
        'locationRestriction': {
            'circle': {
                'center': {'latitude': CENTER_LAT, 'longitude': CENTER_LON},
                'radius': RADIUS_METERS
            }
        }
    }
    headers = {
        'Content-Type': 'application/json',
        'X-Goog-Api-Key': GOOGLE_API_KEY,
        'X-Goog-FieldMask': GOOGLE_NEARBY_FIELD_MASK
    }
    r = requests.post(GOOGLE_NEARBY_URL, headers=headers, json=payload, timeout=30)
    if r.status_code != 200:
        print(r.status_code, r.text[:1000])
    r.raise_for_status()
    return r.json()


def flatten_google_place(place: Dict, query_type: Optional[str] = None) -> Dict:
    loc = place.get('location', {})
    display = place.get('displayName', {}) or {}
    return {
        'source': 'google_places',
        'source_id': place.get('id'),
        'name': display.get('text'),
        'category_original': first_nonempty(place.get('primaryType'), ','.join(place.get('types', [])[:3]) if place.get('types') else None),
        'query_type': query_type,
        'lat': loc.get('latitude'),
        'lon': loc.get('longitude'),
        'address': place.get('formattedAddress'),
        'business_status': place.get('businessStatus'),
        'rating': place.get('rating'),
        'review_count': place.get('userRatingCount'),
        'price': place.get('priceLevel'),
        'website': place.get('websiteUri'),
        'maps_url': place.get('googleMapsUri'),
        'opening_hours_raw': json.dumps(place.get('regularOpeningHours'), ensure_ascii=False) if place.get('regularOpeningHours') else None,
        'current_opening_hours_raw': json.dumps(place.get('currentOpeningHours'), ensure_ascii=False) if place.get('currentOpeningHours') else None,
        'raw': json.dumps(place, ensure_ascii=False)
    }

# Choose a small list to keep API calls manageable.
google_types = ['restaurant', 'cafe', 'library', 'park']
google_rows = []

if GOOGLE_API_KEY:
    for t in google_types:
        print('Querying Google type:', t)
        result = google_nearby_search(t, max_result_count=20)
        save_json(result, RAW_DIR / f'google_nearby_{t}.json')
        for p in result.get('places', []):
            google_rows.append(flatten_google_place(p, query_type=t))
        time.sleep(0.2)
else:
    print('Skipping Google because no API key was provided.')

# Deduplicate by source_id.
google_df = pd.DataFrame(google_rows)
if len(google_df):
    google_df = google_df.drop_duplicates(subset=['source_id'])
    google_df.to_csv(OUT_DIR / 'google_places_flat.csv', index=False)
    save_geojson(google_df.to_dict('records'), OUT_DIR / 'google_places.geojson')

google_df.head()

In [ ]:
if len(google_df):
    display(google_df[['name', 'category_original', 'rating', 'review_count', 'business_status', 'address']].head(10))
    plot_counts(google_df, 'category_original', 'Google Places: top original categories')
    quick_map(google_df)

## Optional: Google Place Details

Use Place Details when you already have a place ID and want more fields. This can trigger different billing tiers depending on the fields requested.

For class, request a small number of records and a small field mask. You can expand the mask after checking the documentation and understanding billing.


In [ ]:
GOOGLE_DETAILS_FIELD_MASK = ','.join([
    'id',
    'displayName',
    'formattedAddress',
    'location',
    'primaryType',
    'types',
    'businessStatus',
    'rating',
    'userRatingCount',
    'regularOpeningHours',
    'accessibilityOptions',
    'websiteUri',
    'googleMapsUri'
])


def google_place_details(place_id: str) -> Dict:
    if not GOOGLE_API_KEY:
        raise ValueError('No Google API key provided')
    url = GOOGLE_DETAILS_URL.format(place_id=place_id)
    headers = {
        'Content-Type': 'application/json',
        'X-Goog-Api-Key': GOOGLE_API_KEY,
        'X-Goog-FieldMask': GOOGLE_DETAILS_FIELD_MASK
    }
    r = requests.get(url, headers=headers, timeout=30)
    if r.status_code != 200:
        print(r.status_code, r.text[:1000])
    r.raise_for_status()
    return r.json()

# Optional, limited test on the first 3 Google results.
run_google_details = False
if run_google_details and len(google_df):
    details = []
    for pid in google_df['source_id'].head(3):
        d = google_place_details(pid)
        details.append(d)
        time.sleep(0.2)
    save_json(details, RAW_DIR / 'google_place_details_sample.json')
    details[:1]
else:
    print('Set run_google_details = True to run a small Place Details sample.')

# 03. Foursquare Places API

Foursquare is useful for querying venues with rich metadata and, depending on access and availability, behavioral proxy fields like `popularity` and `hours_popular`.

Useful fields to test:

```text
fsq_id,name,geocodes,location,categories,chains,closed_bucket,timezone,hours,hours_popular,rating,stats,popularity,price,tastes,features,venue_reality_bucket
```

Some fields may not be available for every account, every place, or every result. Treat missing fields as part of the analysis.


In [ ]:
# FOURSQUARE_API_KEY is defined in the consolidated API key setup cell above.
FOURSQUARE_SEARCH_URL = 'https://api.foursquare.com/v3/places/search'
FOURSQUARE_FIELDS = ','.join([
    'fsq_id', 'name', 'geocodes', 'location', 'categories', 'chains',
    'closed_bucket', 'timezone', 'hours', 'hours_popular', 'rating',
    'stats', 'popularity', 'price', 'tastes', 'features', 'venue_reality_bucket'
])
print(FOURSQUARE_FIELDS)

In [ ]:
# FOURSQUARE_API_KEY is defined in the consolidated API key setup cell above.
def foursquare_search(query: Optional[str] = None, sort: str = 'RELEVANCE', limit: int = 50) -> Dict:
    if not FOURSQUARE_API_KEY:
        raise ValueError('No Foursquare API key provided')
    params = {
        'll': f'{CENTER_LAT},{CENTER_LON}',
        'radius': RADIUS_METERS,
        'limit': min(limit, 50),
        'sort': sort,
        'fields': FOURSQUARE_FIELDS
    }
    if query:
        params['query'] = query
    headers = {
        'Accept': 'application/json',
        'Authorization': FOURSQUARE_API_KEY,
        'X-Places-Api-Version': '1970-01-01'
    }
    r = requests.get(FOURSQUARE_SEARCH_URL, headers=headers, params=params, timeout=30)
    if r.status_code != 200:
        print(r.status_code, r.text[:1000])
    r.raise_for_status()
    return r.json()


def flatten_foursquare_place(place: Dict, query: Optional[str] = None) -> Dict:
    geocodes = place.get('geocodes', {}) or {}
    main = geocodes.get('main', {}) or {}
    categories = place.get('categories') or []
    category_names = [c.get('name') for c in categories if c.get('name')]
    location = place.get('location', {}) or {}
    address = first_nonempty(
        location.get('formatted_address'),
        ', '.join([str(location.get(k)) for k in ['address', 'locality', 'region', 'postcode'] if location.get(k)])
    )
    return {
        'source': 'foursquare',
        'source_id': place.get('fsq_id'),
        'name': place.get('name'),
        'category_original': ', '.join(category_names) if category_names else None,
        'query': query,
        'lat': main.get('latitude'),
        'lon': main.get('longitude'),
        'address': address,
        'closed_bucket': place.get('closed_bucket'),
        'rating': place.get('rating'),
        'review_count': safe_get(place, ['stats', 'total_ratings']),
        'price': place.get('price'),
        'popularity': place.get('popularity'),
        'hours_raw': json.dumps(place.get('hours'), ensure_ascii=False) if place.get('hours') else None,
        'popular_hours_raw': json.dumps(place.get('hours_popular'), ensure_ascii=False) if place.get('hours_popular') else None,
        'venue_reality_bucket': place.get('venue_reality_bucket'),
        'raw': json.dumps(place, ensure_ascii=False)
    }

# Try broad query first, then optional specific terms.
foursquare_queries = [None, 'coffee', 'library', 'park']
foursquare_rows = []

if FOURSQUARE_API_KEY:
    for q in foursquare_queries:
        print('Querying Foursquare:', q or 'broad nearby search')
        result = foursquare_search(query=q, sort='RELEVANCE', limit=50)
        save_json(result, RAW_DIR / f'foursquare_search_{q or "nearby"}.json')
        for p in result.get('results', []):
            foursquare_rows.append(flatten_foursquare_place(p, query=q))
        time.sleep(0.2)
else:
    print('Skipping Foursquare because no API key was provided.')

foursquare_df = pd.DataFrame(foursquare_rows)
if len(foursquare_df):
    foursquare_df = foursquare_df.drop_duplicates(subset=['source_id'])
    foursquare_df.to_csv(OUT_DIR / 'foursquare_flat.csv', index=False)
    save_geojson(foursquare_df.to_dict('records'), OUT_DIR / 'foursquare_places.geojson')

foursquare_df.head()

In [ ]:
if len(foursquare_df):
    display(foursquare_df[['name', 'category_original', 'rating', 'popularity', 'closed_bucket', 'address']].head(10))
    plot_counts(foursquare_df, 'category_original', 'Foursquare: top original categories')
    if 'popularity' in foursquare_df.columns and foursquare_df['popularity'].notna().any():
        foursquare_df['popularity'].plot(kind='hist', bins=20, figsize=(7, 4), title='Foursquare popularity distribution')
        plt.xlabel('Popularity score')
        plt.tight_layout()
        plt.show()
    quick_map(foursquare_df)

# 04. OpenStreetMap via Overpass

OpenStreetMap does not have one universal POI field. POI-like features are usually queried through tags such as:

```text
amenity, shop, tourism, leisure, healthcare, office, craft, public_transport, railway
```

This section queries Overpass directly with Python. You can also paste the query into https://overpass-turbo.eu/ and export GeoJSON manually.


In [ ]:
OVERPASS_URL = 'https://overpass-api.de/api/interpreter'

osm_keys = ['amenity', 'shop', 'tourism', 'leisure', 'healthcare', 'office', 'craft']


def build_overpass_query(bbox, keys=osm_keys):
    min_lon, min_lat, max_lon, max_lat = bbox
    bbox_str = f'{min_lat},{min_lon},{max_lat},{max_lon}'
    parts = []
    for key in keys:
        parts.append(f'node["{key}"]({bbox_str});')
        parts.append(f'way["{key}"]({bbox_str});')
        parts.append(f'relation["{key}"]({bbox_str});')
    query = '[out:json][timeout:25];\n(\n  ' + '\n  '.join(parts) + '\n);\nout center tags;'
    return query

osm_query = build_overpass_query(BBOX)
print(osm_query[:1200])

In [ ]:
def run_overpass(query: str) -> Dict:
    r = requests.post(OVERPASS_URL, data={'data': query}, timeout=60)
    if r.status_code != 200:
        print(r.status_code, r.text[:1000])
    r.raise_for_status()
    return r.json()


def flatten_osm_element(el: Dict) -> Dict:
    tags = el.get('tags', {}) or {}
    lat = el.get('lat') or safe_get(el, ['center', 'lat'])
    lon = el.get('lon') or safe_get(el, ['center', 'lon'])
    original_parts = []
    for k in osm_keys:
        if tags.get(k):
            original_parts.append(f'{k}={tags.get(k)}')
    return {
        'source': 'openstreetmap',
        'source_id': f"{el.get('type')}/{el.get('id')}",
        'osm_type': el.get('type'),
        'name': tags.get('name'),
        'category_original': '; '.join(original_parts) if original_parts else None,
        'lat': lat,
        'lon': lon,
        'address': first_nonempty(
            tags.get('addr:full'),
            ' '.join([str(tags.get(k)) for k in ['addr:housenumber', 'addr:street'] if tags.get(k)])
        ),
        'website': first_nonempty(tags.get('website'), tags.get('contact:website')),
        'phone': first_nonempty(tags.get('phone'), tags.get('contact:phone')),
        'opening_hours': tags.get('opening_hours'),
        'raw_tags': json.dumps(tags, ensure_ascii=False),
        'raw': json.dumps(el, ensure_ascii=False)
    }

run_osm = True
osm_rows = []
if run_osm:
    osm_result = run_overpass(osm_query)
    save_json(osm_result, RAW_DIR / 'osm_overpass_raw.json')
    osm_rows = [flatten_osm_element(el) for el in osm_result.get('elements', [])]
else:
    print('Set run_osm = True to query Overpass')

osm_df = pd.DataFrame(osm_rows)
if len(osm_df):
    osm_df = osm_df.drop_duplicates(subset=['source_id'])
    osm_df.to_csv(OUT_DIR / 'osm_flat.csv', index=False)
    save_geojson(osm_df.to_dict('records'), OUT_DIR / 'osm_places.geojson')

osm_df.head()

In [ ]:
if len(osm_df):
    display(osm_df[['name', 'category_original', 'opening_hours', 'website', 'address']].head(10))
    plot_counts(osm_df, 'category_original', 'OpenStreetMap: top original tag combinations', top_n=20)
    quick_map(osm_df)

# 05. Overture Maps Places

Overture Maps distributes open places data as cloud-hosted GeoParquet. This is different from calling a search API: you can query or download the data you need and build your own platform logic from it.

The current Places Guide describes the `place` feature type as point representations of real-world entities. Key fields include `id`, `names`, `categories`, `confidence`, `sources`, `websites`, `socials`, `emails`, `phones`, `addresses`, `operating_status`, and geometry.

There are two good ways to work with Overture in class:

1. Use the `overturemaps` Python client or command-line tool.
2. Use DuckDB to query GeoParquet directly.

The Python client is simpler for students. DuckDB is better for teaching data infrastructure.


In [ ]:
# Option A: Overture Python client / command-line tool.
# In Colab you may need: !pip -q install overturemaps

import subprocess

bbox_str = ','.join(map(str, BBOX))
overture_geojson_path = OUT_DIR / 'overture_places.geojson'

cmd = [
    'overturemaps', 'download',
    '--bbox', bbox_str,
    '-f', 'geojson',
    '--type', 'place',
    '-o', str(overture_geojson_path)
]

print('Command to run:')
print(' '.join(cmd))

run_overture_cli = False
if run_overture_cli:
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
else:
    print('Set run_overture_cli = True to download Overture Places for the bbox.')

In [ ]:
# Option B: DuckDB SQL query against Overture GeoParquet.
# In Colab you may need: !pip -q install duckdb

DUCKDB_SQL = f"""
LOAD spatial;
LOAD httpfs;
SET s3_region='us-west-2';

COPY (
  SELECT
    id,
    version,
    names.primary AS name,
    categories.primary AS category,
    basic_category,
    confidence,
    operating_status,
    CAST(sources AS JSON) AS sources,
    CAST(websites AS JSON) AS websites,
    CAST(addresses AS JSON) AS addresses,
    geometry
  FROM read_parquet(
    's3://overturemaps-us-west-2/release/2026-06-17.0/theme=places/type=place/*',
    filename=true,
    hive_partitioning=1
  )
  WHERE
    bbox.xmin BETWEEN {BBOX[0]} AND {BBOX[2]}
    AND bbox.ymin BETWEEN {BBOX[1]} AND {BBOX[3]}
    AND confidence > 0.60
) TO '{overture_geojson_path}' WITH (FORMAT GDAL, DRIVER 'GeoJSON', SRS 'EPSG:4326');
"""
print(DUCKDB_SQL[:1600])

# To run in a local DuckDB connection, uncomment the block below.
# import duckdb
# con = duckdb.connect()
# con.execute('INSTALL spatial;')
# con.execute('INSTALL httpfs;')
# con.execute(DUCKDB_SQL)
# con.close()


In [ ]:
def load_overture_geojson(path: Path) -> pd.DataFrame:
    if not path.exists():
        print('No Overture file found yet:', path)
        return pd.DataFrame()
    gj = load_json(path)
    rows = []
    for f in gj.get('features', []):
        props = f.get('properties', {}) or {}
        coords = safe_get(f, ['geometry', 'coordinates'], [None, None])
        rows.append({
            'source': 'overture',
            'source_id': props.get('id'),
            'name': props.get('name') or safe_get(props, ['names', 'primary']),
            'category_original': first_nonempty(props.get('category'), props.get('basic_category'), safe_get(props, ['categories', 'primary'])),
            'lat': coords[1] if coords else None,
            'lon': coords[0] if coords else None,
            'confidence': props.get('confidence'),
            'operating_status': props.get('operating_status'),
            'address': props.get('address'),
            'sources_raw': json.dumps(props.get('sources'), ensure_ascii=False) if props.get('sources') else None,
            'raw': json.dumps(props, ensure_ascii=False)
        })
    return pd.DataFrame(rows)

overture_df = load_overture_geojson(overture_geojson_path)
if len(overture_df):
    overture_df.to_csv(OUT_DIR / 'overture_flat.csv', index=False)

overture_df.head()

In [ ]:
if len(overture_df):
    display(overture_df[['name', 'category_original', 'confidence', 'operating_status']].head(10))
    plot_counts(overture_df, 'category_original', 'Overture: top original categories', top_n=20)
    if 'confidence' in overture_df.columns and overture_df['confidence'].notna().any():
        overture_df['confidence'].plot(kind='hist', bins=20, figsize=(7,4), title='Overture confidence distribution')
        plt.xlabel('Confidence')
        plt.tight_layout()
        plt.show()
    quick_map(overture_df)

# 06. Normalize categories across sources

Normalization is an interpretive act. Preserve each source's original category, then add a standardized field that helps you compare.

Edit the function below as your categories become more specific.


In [ ]:
def standardize_category(cat: Any) -> str:
    if pd.isna(cat):
        return 'unknown'
    c = str(cat).lower()
    food_terms = ['restaurant', 'cafe', 'coffee', 'bar', 'bakery', 'food', 'pizza', 'deli', 'drink']
    retail_terms = ['shop', 'store', 'retail', 'supermarket', 'grocery', 'market', 'pharmacy']
    culture_terms = ['museum', 'gallery', 'theatre', 'theater', 'library', 'cultural', 'arts']
    education_terms = ['school', 'college', 'university', 'education', 'kindergarten']
    health_terms = ['hospital', 'clinic', 'doctor', 'dentist', 'healthcare', 'pharmacy']
    religion_terms = ['church', 'mosque', 'synagogue', 'temple', 'religion', 'place_of_worship']
    transit_terms = ['transit', 'subway', 'bus', 'railway', 'station']
    park_terms = ['park', 'garden', 'playground', 'open_space', 'recreation']
    service_terms = ['bank', 'atm', 'laundry', 'post_office', 'service', 'office']
    government_terms = ['government', 'courthouse', 'police', 'fire_station', 'townhall']

    tests = [
        ('food_drink', food_terms),
        ('retail', retail_terms),
        ('culture', culture_terms),
        ('education', education_terms),
        ('health', health_terms),
        ('religion', religion_terms),
        ('transit', transit_terms),
        ('park_open_space', park_terms),
        ('service', service_terms),
        ('government', government_terms),
    ]
    for label, terms in tests:
        if any(t in c for t in terms):
            return label
    return 'other'

source_dfs = []
for df in [google_df, foursquare_df, osm_df, overture_df]:
    if len(df):
        source_dfs.append(df.copy())

combined_df = pd.concat(source_dfs, ignore_index=True, sort=False) if source_dfs else pd.DataFrame()
if len(combined_df):
    combined_df['category_standardized'] = combined_df['category_original'].apply(standardize_category)
    keep_cols = [
        'source', 'source_id', 'name', 'category_original', 'category_standardized',
        'lat', 'lon', 'address', 'website', 'phone', 'opening_hours', 'opening_hours_raw',
        'rating', 'review_count', 'price', 'popularity', 'popular_hours_raw',
        'confidence', 'business_status', 'operating_status', 'closed_bucket', 'notes'
    ]
    for col in keep_cols:
        if col not in combined_df.columns:
            combined_df[col] = None
    combined_export = combined_df[keep_cols].copy()
    combined_export.to_csv(OUT_DIR / 'combined_poi_normalized.csv', index=False)
    save_geojson(combined_export.to_dict('records'), OUT_DIR / 'combined_poi_normalized.geojson')

combined_df.head()

In [ ]:
if len(combined_df):
    display(combined_df[['source', 'name', 'category_original', 'category_standardized']].head(20))
    plot_counts(combined_df, 'source', 'Records by source')
    plot_counts(combined_df, 'category_standardized', 'Records by standardized category')
    quick_map(combined_df)

# 07. Compare metadata availability

Instead of asking only "which source has more points?" ask "which source has which kinds of metadata?"

This section creates a simple metadata availability table. A value counts as available if it is not null and not an empty string.


In [ ]:
metadata_fields = [
    'address', 'website', 'phone', 'opening_hours', 'opening_hours_raw',
    'rating', 'review_count', 'price', 'popularity', 'popular_hours_raw',
    'confidence', 'business_status', 'operating_status', 'closed_bucket'
]

def availability_by_source(df: pd.DataFrame, fields: List[str]) -> pd.DataFrame:
    rows = []
    for source, sub in df.groupby('source'):
        total = len(sub)
        row = {'source': source, 'record_count': total}
        for field in fields:
            if field in sub.columns:
                available = sub[field].notna() & (sub[field].astype(str).str.len() > 0) & (sub[field].astype(str) != 'None')
                row[field] = int(available.sum())
                row[f'{field}_pct'] = round(100 * available.sum() / total, 1) if total else 0
            else:
                row[field] = 0
                row[f'{field}_pct'] = 0
        rows.append(row)
    return pd.DataFrame(rows)

if len(combined_df):
    avail = availability_by_source(combined_df, metadata_fields)
    display(avail[['source', 'record_count'] + [f'{f}_pct' for f in metadata_fields if f in combined_df.columns]].round(1))
    avail.to_csv(OUT_DIR / 'metadata_availability_by_source.csv', index=False)
else:
    print('No combined data yet.')

In [ ]:
if len(combined_df):
    # Build a long-form table for plotting.
    pct_cols = [c for c in avail.columns if c.endswith('_pct')]
    long_avail = avail.melt(id_vars='source', value_vars=pct_cols, var_name='field', value_name='percent_available')
    long_avail['field'] = long_avail['field'].str.replace('_pct', '', regex=False)
    display(long_avail.head())

    # Simple chart: one source at a time. Change selected_source.
    selected_source = long_avail['source'].iloc[0]
    plot_df = long_avail[long_avail['source'] == selected_source].sort_values('percent_available')
    ax = plot_df.plot(x='field', y='percent_available', kind='barh', legend=False, figsize=(8, 5), title=f'Metadata availability: {selected_source}')
    ax.set_xlabel('Percent of records with field available')
    ax.set_ylabel('Field')
    plt.tight_layout()
    plt.show()

# 08. Source comparison prompts

Write directly in this notebook or in a separate document.

## Prompt 1: What appears?

Which places appear across multiple platforms? Which places appear in only one? What might explain the difference?

## Prompt 2: What categories dominate?

Which categories are most common in each source? What does that suggest about the platform's model of place?

## Prompt 3: What metadata changes your understanding?

Does ratings data, popular hours, confidence, operating status, opening hours, or source attribution change how you understand this area?

## Prompt 4: What is missing?

List at least five places, uses, practices, atmospheres, histories, or social relations that matter in the study area but do not appear in the data.

## Prompt 5: What would you need to make yourself?

Would you need field observation, sensing, interviews, OCR, imagery, archival work, participatory mapping, or semantic annotation to represent what is missing?


In [ ]:
reflection_template = f"""
# POI Platform Audit Reflection

Study area: {STUDY_AREA_NAME}
Bounding box: {BBOX}

## 1. What does each source think a place is?

Google Places:

Foursquare:

OpenStreetMap:

Overture Maps:

## 2. What appears and what is missing?

Places that appeared across multiple systems:

Places that appeared in only one system:

Places that were missing from all systems:

## 3. Metadata critique

Fields that changed how I understood the area:

Fields I wanted but could not access:

Fields that felt ethically or politically sensitive:

## 4. Prototype translation

My final platform might use POI data to:

POI data would be insufficient because:

Fields I would need to create myself:

Potential risks of making these places visible:
"""

reflection_path = OUT_DIR / 'poi_platform_audit_reflection_template.md'
reflection_path.write_text(reflection_template, encoding='utf-8')
print(reflection_template)
print('Saved:', reflection_path)

# 09. Export checklist

Before submitting, make sure you have:

- Raw files from each source you queried in `poi_data/raw/`.
- Flattened source CSVs in `poi_data/output/`.
- `combined_poi_normalized.csv`.
- `combined_poi_normalized.geojson`.
- `metadata_availability_by_source.csv`.
- A short reflection using the template above.
- One screenshot or exported map showing your study area and points.

## Final critical reminder

Do not write: "Google shows the real places" or "OSM is the accurate version."

Instead write: "This source makes some places visible through a particular schema, infrastructure, and user model."
